# 5. YOLO detector (future work, not used in the current scope)
Fine-tunes a YOLO detection model on the scenes from notebook 04. Set `YOLO_FILE=yolo_best.pt` in `.env` so the API serves it (it then returns boxes and joins the labels left to right).

In [ ]:
import os, sys
from pathlib import Path

# run everything from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))  # so `backend.ml.preprocessing` can be imported
print("project root:", ROOT)

In [ ]:
# ---- settings ----
DATA = "data/processed/yolo_scenes/data.yaml"
WEIGHTS = "yolov8n.pt"      # or yolo11n.pt
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = None               # e.g. 0 for GPU, "cpu"
SEED = 42
OUT = "models/yolo_best.pt"
REPORT_DIR = "reports/yolo"

In [ ]:
import json, shutil

from ultralytics import YOLO

model = YOLO(WEIGHTS)
model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
            project="runs/yolo", name="modi", exist_ok=True, patience=20, seed=SEED)

best = Path(model.trainer.save_dir) / "weights" / "best.pt"
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, OUT)

In [ ]:
m = YOLO(OUT).val(data=DATA, split="test", imgsz=IMGSZ, device=DEVICE)
metrics = {"map50": float(m.box.map50), "map50_95": float(m.box.map),
           "precision": float(m.box.mp), "recall": float(m.box.mr)}
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
(Path(REPORT_DIR) / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("test metrics:", metrics)
print(f"saved {OUT}")